# Análise do Impacto da Variação do Preço do Combustível no Preço dos Alimentos: Caso de Estudo do Milho Branco

O presente projecto tem como objectivo investigar de que forma as alterações nos preços dos combustíveis influenciam o custo do milho branco no mercado moçambicano, utilizando técnicas de Análise Exploratória de Dados (EDA). 

O estudo baseia-se em conjuntos de dados contendo informações históricas sobre preços de combustíveis e preços do milho branco, onde são formuladas hipóteses estatísticas relacionadas ao impacto do combustível no preço do milho branco, incluindo testes de significância para validar as relações observadas nos dados.

Por fim, o estudo sintetiza os principais achados, conclusões e recomendações para futuras análises, contribuindo para o apoio à tomada de decisão económica e social relacionada aos preços dos alimentos e combustíveis.

## Objectivos Específicos

* Analisar o impacto da variação do preço do combustível no preço do milho branco em Moçambique.
* Identificar padrões, tendências e relações entre os preços dos combustíveis e dos alimentos ao longo do tempo.
* Explorar o conjunto de dados através de técnicas de Análise Exploratória de Dados (EDA).
* Realizar a limpeza, transformação e preparação dos dados para garantir a qualidade da análise.
* Aplicar técnicas de engenharia de recursos e visualização de dados para melhorar a interpretação dos resultados.
* Formular e testar hipóteses estatísticas relacionadas à influência do combustível no preço do milho branco.
* Avaliar a significância estatística das relações identificadas nos dados.
* Produzir insights e conclusões que possam apoiar decisões económicas e políticas relacionadas aos preços dos alimentos e combustíveis.


---
# Código

In [143]:
import os
import pandas as pd
import numpy as np

from collections import Counter

import warnings
warnings.filterwarnings('ignore')


In [150]:
def analisar_qualidade(df, df_caminho):
    # Dimensões básicas
    total_linhas = df.shape[0]
    total_colunas = df.shape[1]
    total_celulas = total_linhas * total_colunas
    
    # Tamanho em memória
    tamanho_mb_dico = os.path.getsize(df_caminho) / 1024**2
    tamanho_mb_memoria = df.memory_usage(deep=True).sum() / (1024 ** 2)
    
    
    # Valores ausentes
    valores_ausentes = df.isnull().sum().sum()
    percentagem_ausentes = (valores_ausentes / total_celulas) * 100
    
    # Duplicados
    duplicados = df.duplicated().sum()
    percentagem_duplicados = (duplicados / total_linhas) * 100
    
    # Dados válidos
    dados_validos = total_celulas - valores_ausentes
    percentagem_validos = (dados_validos / total_celulas) * 100
    
    # Linhas completas (sem valores ausentes)
    linhas_completas = df.dropna().shape[0]
    percentagem_linhas_completas = (linhas_completas / total_linhas) * 100
    
    # Linhas incompletas (com valores ausentes)
    linhas_incompletas = total_linhas - linhas_completas
    percentagem_linhas_incompletas = (linhas_incompletas / total_linhas) * 100
    
    # =====================================================
    # EXIBIÇÃO DE RESULTADOS
    # =====================================================
    
    print("=" * 70)
    print("ANÁLISE DE QUALIDADE DO DATASET")
    print("=" * 70)
    
    # Detalhes gerais
    print("\n DETALHES GERAIS")
    print("-" * 70)
    print(f"Total de linhas: {total_linhas:,}")
    print(f"Total de colunas: {total_colunas}")
    print(f"Total de células: {total_celulas:,}")
    print(f"Tamanho do dataset (em disco): {tamanho_mb_dico:.3f} MB")
    print(f"Tamanho do dataset (em memória): {tamanho_mb_memoria:.3f} MB")
    
    # Tabela de métricas principais
    print("\n RESUMO DA QUALIDADE DOS DADOS")
    print("-" * 70)
    
    metricas = pd.DataFrame({
        "Métrica": [
            "Valores ausentes",
            "Duplicados",
            "Dados válidos",
            "Linhas completas",
            "Linhas incompletas"
        ],
        "Quantidade": [
            valores_ausentes,
            duplicados,
            dados_validos,
            linhas_completas,
            linhas_incompletas
        ],
        "Percentagem (%)": [
            f"{percentagem_ausentes:.1f}%",
            f"{percentagem_duplicados:.1f}%",
            f"{percentagem_validos:.1f}%",
            f"{percentagem_linhas_completas:.1f}%",
            f"{percentagem_linhas_incompletas:.1f}%"
        ]
    })
    
    print(metricas.to_string(index=False))
    
    # Valores ausentes por coluna
    print("\n VALORES AUSENTES POR COLUNA")
    print("-" * 70)
    
    ausentes_coluna = df.isnull().sum()
    percentagem_coluna = (ausentes_coluna / total_linhas) * 100
    
    tabela_ausentes = pd.DataFrame({
        "Coluna": df.columns,
        "Valores Ausentes": ausentes_coluna.values,
        "Percentagem (%)": [f"{p:.1f}%" for p in percentagem_coluna.values]
    })
    
    print(tabela_ausentes.to_string(index=False))
    
    # Tipos de dados
    print("\n TIPOS DE DADOS")
    print("-" * 70)
    print(df.dtypes)
    
    # Estatísticas descritivas (apenas colunas numéricas)
    print("\n ESTATÍSTICAS DESCRITIVAS (COLUNAS NUMÉRICAS)")
    print("-" * 70)
    print(df.describe().to_string())




def valores_em_falta(df):

    # Função para formatar milhares com ponto
    fmt = lambda x: f"{x:,}".replace(",", ".")

    percentagens = {}

    # Espaço total esperado
    provincias = df['provincia'].unique()
    anos = df['ano'].unique()
    meses = df['mes'].unique()

    total_esperado = (
        len(provincias) *
        len(anos) *
        len(meses)
    )

    # Total real por PUTP
    reais = (
        df.groupby('PUTP')
        .size()
    )

    # Calcular percentagem em falta
    for putp, total_real in reais.items():

        falta = total_esperado - total_real

        perc_falta = (
            falta / total_esperado
        ) * 100

        if perc_falta > 0:
            percentagens[putp] = perc_falta

    # Ordenar da maior para a menor percentagem
    percentagens = dict(
        sorted(
            percentagens.items(),
            key=lambda x: x[1],
            reverse=True
        )
    )

    # Imprimir
    print("=" * 75)
    print("PERCENTAGEM DE AMOSTRAS EM FALTA POR PRODUTO")
    print("=" * 75)

    for produto, perc in percentagens.items():
        print(f"{produto:<55} {perc:>8.2f}%")

    print()
    print("=" * 75)
    print(f"Total esperado por PUTP: {fmt(total_esperado)}")
    # print(f"Produtos com faltas    : {fmt(len(percentagens))}")
    print("=" * 75)

In [144]:
caminho_food = 'data/raw/wfp_food_prices_moz 20260508.csv'
caminho_combustivel = 'data/processed/Evolucao-de-Precos-dos-Produtos-Petroliferos-2010-2026.csv'

---
## Preço do Combustível 

**Contexto:** O dataset contém os preços mensais da **gasolina** e do **gasóleo** em Moçambique.

**Período:** Janeiro de 2010 á Maio de 2026. 

**Fonte:** [**ARENE** (Autoridade Reguladora de Energia).](https://arene.org.mz/combustiveis-liquidos/estatisticas/) 

**Nota:** Originalmente, os dados encontravam-se em formato PDF, tendo sido necessário realizar um processo conversão para o formato CSV para facilitar o uso.

In [145]:
precos_combustivel = pd.read_csv(caminho_combustivel)

In [146]:
precos_combustivel["Gasolina"] = precos_combustivel["Gasolina"].str.replace(",", ".").astype(float)
precos_combustivel["Gasóleo"] = precos_combustivel["Gasóleo"].str.replace(",", ".").astype(float)

In [147]:
precos_combustivel.head()

,Ano,Mês,Gasolina,Gasóleo
0,2010,1,23.10,22.45
1,2010,2,23.10,22.45
2,2010,3,23.10,22.45
3,2010,4,26.57,24.70
4,2010,5,31.09,28.16


---
### Dicionário de Dados

**Ano**

* **Tipo de dado:** Inteiro (`int`)
* **Descrição:** Ano de referência do registo dos preços dos combustíveis.

**Mês**

* **Tipo de dado:** Inteiro (`int`)
* **Descrição:** Mês correspondente ao registo mensal dos preços dos combustíveis.

**Gasolina**

* **Tipo de dado:** Decimal (`float`)
* **Descrição:** Preço mensal da gasolina em Moçambique, expresso em Meticais por litro (MZN/L).

**Gasóleo**

* **Tipo de dado:** Decimal (`float`)
* **Descrição:** Preço mensal do gasóleo em Moçambique, expresso em Meticais por litro (MZN/L).


In [151]:
analisar_qualidade(precos_combustivel, caminho_combustivel)

ANÁLISE DE QUALIDADE DO DATASET

 DETALHES GERAIS
----------------------------------------------------------------------
Total de linhas: 197
Total de colunas: 4
Total de células: 788
Tamanho do dataset (em disco): 0.005 MB
Tamanho do dataset (em memória): 0.006 MB

 RESUMO DA QUALIDADE DOS DADOS
----------------------------------------------------------------------
           Métrica  Quantidade Percentagem (%)
  Valores ausentes           0            0.0%
        Duplicados           0            0.0%
     Dados válidos         788          100.0%
  Linhas completas         197          100.0%
Linhas incompletas           0            0.0%

 VALORES AUSENTES POR COLUNA
----------------------------------------------------------------------
  Coluna  Valores Ausentes Percentagem (%)
     Ano                 0            0.0%
     Mês                 0            0.0%
Gasolina                 0            0.0%
 Gasóleo                 0            0.0%

 TIPOS DE DADOS
----------------

## Preço de Alimentos

**Contexto:** O dataset contém os preços mensais de vários produtos alimentares no mercado moçambicano em metical (MZN) e dólar norte-americano (USD).

**Período:** Novembro de 1992 á Dezembro de 2025.

**Fonte:** [**World Food Programme** (através da HDX).](https://data.humdata.org/dataset/wfp-food-prices-for-mozambique)


In [152]:
food_price = pd.read_csv(caminho_food)

In [153]:
food_price.head()

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,1992-11-15,Maputo City,Cidade_De_Maputo,Maputo,321,-25.97,32.59,cereals and tubers,Maize (white),67,KG,actual,Retail,MZN,1.34,0.046
1,1992-12-15,Gaza,Chokwe,Chokwe,317,-24.53,32.98,cereals and tubers,Maize (white),67,KG,actual,Retail,MZN,1.53,0.053
2,1992-12-15,Inhambane,Maxixe,Maxixe,331,-23.86,35.35,cereals and tubers,Maize (white),67,KG,actual,Retail,MZN,1.69,0.058
3,1992-12-15,Maputo City,Cidade_De_Maputo,Maputo,321,-25.97,32.59,cereals and tubers,Maize (white),67,KG,actual,Retail,MZN,1.55,0.053
4,1993-01-15,Gaza,Chokwe,Chokwe,317,-24.53,32.98,cereals and tubers,Maize (white),67,KG,actual,Retail,MZN,1.67,0.058


---
### Dicionário de Dados

**date**

* **Tipo de dado:** Data (`date`)
* **Descrição:** Data do registo do preço do alimento no mercado.

**admin1**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Província onde o mercado está localizado.

**admin2**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Distrito do mercado.

**market**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Nome do mercado onde o preço foi recolhido.

**market_id**

* **Tipo de dado:** Inteiro (`int`)
* **Descrição:** Identificador único do mercado.

**latitude**

* **Tipo de dado:** Decimal (`float`)
* **Descrição:** Coordenada geográfica de latitude do mercado.

**longitude**

* **Tipo de dado:** Decimal (`float`)
* **Descrição:** Coordenada geográfica de longitude do mercado.

**category**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Categoria do produto alimentar (ex.: cereais, legumes, óleo, etc.).

**commodity**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Nome do produto alimentar comercializado.

**commodity_id**

* **Tipo de dado:** Inteiro (`int`)
* **Descrição:** Identificador único do produto alimentar.

**unit**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Unidade de medida utilizada para o preço do produto (ex.: kg, litro, saco).

**priceflag**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Indicador do estado do preço registado (ex.: efectivo, previsto).

**pricetype**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Tipo de preço registado (ex.: retalho, grosso).

**currency**

* **Tipo de dado:** Texto (`string`)
* **Descrição:** Moeda utilizada no registo do preço.

**price**

* **Tipo de dado:** Decimal (`float`)
* **Descrição:** Preço do produto alimentar em metical (MZN).

**usdprice**

* **Tipo de dado:** Decimal (`float`)
* **Descrição:** Preço do produto convertido para dólares americanos (USD).


In [154]:
food_price['date'] = pd.to_datetime(food_price['date'])

In [155]:
analisar_qualidade(food_price, caminho_food)

ANÁLISE DE QUALIDADE DO DATASET

 DETALHES GERAIS
----------------------------------------------------------------------
Total de linhas: 68,082
Total de colunas: 16
Total de células: 1,089,312
Tamanho do dataset (em disco): 8.294 MB
Tamanho do dataset (em memória): 37.242 MB

 RESUMO DA QUALIDADE DOS DADOS
----------------------------------------------------------------------
           Métrica  Quantidade Percentagem (%)
  Valores ausentes           8            0.0%
        Duplicados           0            0.0%
     Dados válidos     1089304          100.0%
  Linhas completas       68080          100.0%
Linhas incompletas           2            0.0%

 VALORES AUSENTES POR COLUNA
----------------------------------------------------------------------
      Coluna  Valores Ausentes Percentagem (%)
        date                 0            0.0%
      admin1                 2            0.0%
      admin2                 2            0.0%
      market                 0            0.0%
  

## Limpeza do Dataset

In [156]:
food_price[food_price['category'] == 'non-food']['commodity'].unique()

<StringArray>
['Fuel (diesel)', 'Fuel (petrol-gasoline)', 'Firewood', 'Handwash soap']
Length: 4, dtype: str

In [157]:
# remove 'non-food' category data
food_price = food_price.loc[(food_price['category'] != 'non-food')]

In [159]:
food_price['ano'] = food_price['date'].dt.year
food_price['mes'] = food_price['date'].dt.month

food_price = food_price.loc[(food_price['ano'] >= 2000)]

In [160]:
food_price.drop(['date', 'priceflag', 'currency', 'usdprice'], axis=1, inplace=True)

In [161]:
food_price = food_price[
    [
        'ano',
        'mes',
        'admin1',
        'admin2',
        'market',
        'market_id',
        'latitude',
        'longitude',
        'category',
        'commodity',
        'commodity_id',
        'unit',
        'pricetype',
        'price'
    ]
]

# Renomear colunas para português
food_price = food_price.rename(columns={
    'admin1': 'provincia',
    'admin2': 'distrito',
    'market': 'mercado',
    'market_id': 'id_mercado',
    'category': 'categoria',
    'commodity': 'produto',
    'commodity_id': 'id_produto',
    'unit': 'unidade',
    'pricetype': 'tipo_preco',
    'price': 'preco'
})

# Resetar index
food_price = food_price.reset_index(drop=True)

print('food_price new shape: ', food_price.shape)


food_price new shape:  (66053, 14)


In [163]:
# Traduzir valores da coluna categoria
food_price['categoria'] = food_price['categoria'].replace({
    'cereals and tubers': 'cereais e tubérculos',
    'miscellaneous food': 'alimentos diversos',
    'oil and fats': 'óleos e gorduras',
    'pulses and nuts': 'leguminosas e nozes',
    'vegetables and fruits': 'vegetais e frutas',
    'meat, fish and eggs': 'carne, peixe e ovos'
})

# Traduzir valores da coluna produto
food_price['produto'] = food_price['produto'].replace({
    'Rice': 'Arroz',
    'Maize (white)': 'Milho (branco)',
    'Cassava (dry)': 'Mandioca (seca)',
    'Rice (imported)': 'Arroz (importado)',
    'Sugar (brown, local)': 'Açúcar (castanho, local)',
    'Oil (vegetable, local)': 'Óleo vegetal (local)',
    'Beans (dry)': 'Feijão (seco)',
    'Groundnuts (Mix)': 'Amendoim (mistura)',
    'Maize meal (white, first grade)': 'Farinha de milho (branca, primeira)',
    'Maize meal (white, with bran)': 'Farinha de milho (branca, com farelo)',
    'Oil (vegetable, imported)': 'Óleo vegetal (importado)',
    'Cowpeas': 'Feijão nhemba',
    'Groundnuts (large, shelled)': 'Amendoim (grande, descascado)',
    'Groundnuts (small, shelled)': 'Amendoim (pequeno, descascado)',
    'Wheat flour (local)': 'Farinha de trigo (local)',
    'Cassava flour': 'Farinha de mandioca',
    'Cassava (fresh)': 'Mandioca (fresca)',
    'Maize meal (white, without bran)': 'Farinha de milho (branca, sem farelo)',
    'Potatoes (Irish, imported)': 'Batata reno (importada)',
    'Potatoes (Irish, local)': 'Batata reno (local)',
    'Rice (local)': 'Arroz (local)',
    'Sweet potatoes': 'Batata-doce',
    'Cabbage': 'Repolho',
    'Carrots': 'Cenoura',
    'Garlic (large)': 'Alho (grande)',
    'Garlic (small)': 'Alho (pequeno)',
    'Onions (local)': 'Cebola (local)',
    'Peppers (green)': 'Pimento (verde)',
    'Tomatoes': 'Tomate',
    'Onions': 'Cebola',
    'Beans (fresh)': 'Feijão (fresco)',
    'Potatoes (unica)': 'Batata (única)',
    'Maize (imported)': 'Milho (importado)',
    'Beans (red)': 'Feijão (vermelho)',
    'Beans (butter)': 'Feijão manteiga',
    'Beans (catarino)': 'Feijão catarino',
    'Beans (magnum)': 'Feijão magnum',
    'Sugar (brown, imported)': 'Açúcar (castanho, importado)',
    'Onions (imported)': 'Cebola (importada)',
    'Maize meal': 'Farinha de milho',
    'Potatoes': 'Batata',
    'Eggs': 'Ovos',
    'Fish': 'Peixe',
    'Salt (iodised)': 'Sal (iodado)',
    'Sugar': 'Açúcar',
    'Oil (vegetable)': 'Óleo vegetal',
    'Groundnuts': 'Amendoim',
    'Cassava leaves': 'Folhas de mandioca',
    'Coconut': 'Coco',
    'Garlic': 'Alho',
    'Kale': 'Couve',
    'Sesame': 'Gergelim',
    'Soybeans': 'Soja'
})

# Traduzir valores da coluna tipo_preco
food_price['tipo_preco'] = food_price['tipo_preco'].replace({
    'Retail': 'Retalho',
    'Wholesale': 'Grossista'
})

In [164]:
food_price.head()

,ano,mes,provincia,distrito,mercado,id_mercado,latitude,longitude,categoria,produto,id_produto,unidade,tipo_preco,preco
0,2000,1,Cabo_Delgado,Montepuez,Montepuez,333,-13.13,39.00,cereais e tubérculos,Arroz,52,KG,Retalho,7.10
1,2000,1,Gaza,Chokwe,Chokwe,317,-24.53,32.98,cereais e tubérculos,Milho (branco),67,KG,Retalho,3.12
2,2000,1,Gaza,Chokwe,Chokwe,317,-24.53,32.98,cereais e tubérculos,Arroz,52,KG,Retalho,5.59
3,2000,1,Inhambane,Maxixe,Maxixe,331,-23.86,35.35,cereais e tubérculos,Milho (branco),67,KG,Retalho,2.02
4,2000,1,Inhambane,Maxixe,Maxixe,331,-23.86,35.35,cereais e tubérculos,Arroz,52,KG,Retalho,5.20


In [166]:
food_price.to_csv('data/processed/food_price_tratado.csv', index=False)

## Tratamento

In [167]:
food_price = (
    food_price.groupby([
        'provincia',
        'categoria',
        'produto',
        'unidade',
        'tipo_preco',
        'ano',
        'mes'
    ])['preco']
    .mean()
    .round(2)
    .reset_index()
)

In [168]:
food_price.head()

,provincia,categoria,produto,unidade,tipo_preco,ano,mes,preco
0,Cabo_Delgado,alimentos diversos,Açúcar,KG,Retalho,2021,2,93.89
1,Cabo_Delgado,alimentos diversos,Açúcar,KG,Retalho,2021,3,75.00
2,Cabo_Delgado,alimentos diversos,Açúcar,KG,Retalho,2021,4,85.00
3,Cabo_Delgado,alimentos diversos,Açúcar,KG,Retalho,2021,5,73.89
4,Cabo_Delgado,alimentos diversos,Açúcar,KG,Retalho,2021,6,72.78


In [169]:
# Criar identificador único
food_price['PUTP'] = (
    food_price['produto'] + '   [' +
    food_price['unidade'] + '/' +
    food_price['tipo_preco'] + ']'
)


# Produtos
produtos = food_price['produto'].values

# Limite mínimo de ocorrências
limite = 2_000

# Contagem de ocorrências
contagem = Counter(produtos)

# Filtrar produtos
produtos_validos = {
    produto: n
    for produto, n in contagem.items()
    if n >= limite
}

produtos_insuficientes = {
    produto: n
    for produto, n in contagem.items()
    if n < limite
}

# Ordenar do maior para o menor
produtos_validos = dict(
    sorted(produtos_validos.items(), key=lambda x: x[1], reverse=True)
)

produtos_insuficientes = dict(
    sorted(produtos_insuficientes.items(), key=lambda x: x[1])
)

# Função para formatar milhares com ponto
fmt = lambda x: f"{x:,}".replace(",", ".")

# Exibir resultados
print("=" * 55)
print(f"PRODUTOS COM PELO MENOS {fmt(limite)} AMOSTRAS")
print("=" * 55)

for produto, total in produtos_validos.items():
    print(f"{produto:<45} {fmt(total):>8}")

print()
print("=" * 55)

print(f"Produtos suficientes   : {fmt(len(produtos_validos))}")
print(f"Produtos insuficientes : {fmt(len(produtos_insuficientes))}")

PRODUTOS COM PELO MENOS 2.000 AMOSTRAS
Milho (branco)                                   3.201
Arroz                                            2.218
Arroz (importado)                                2.198
Óleo vegetal (local)                             2.162
Açúcar (castanho, local)                         2.153

Produtos suficientes   : 5
Produtos insuficientes : 48


In [170]:
valores_em_falta(food_price)

PERCENTAGEM DE AMOSTRAS EM FALTA POR PRODUTO
Batata (única)   [KG/Retalho]                              99.97%
Cebola (importada)   [KG/Retalho]                          99.97%
Milho (importado)   [KG/Retalho]                           99.97%
Soja   [KG/Retalho]                                        99.53%
Batata reno (importada)   [KG/Retalho]                     99.21%
Gergelim   [KG/Retalho]                                    99.01%
Feijão (fresco)   [KG/Retalho]                             98.98%
Batata reno (local)   [KG/Retalho]                         98.92%
Mandioca (fresca)   [KG/Retalho]                           98.89%
Alho (grande)   [KG/Retalho]                               98.86%
Alho (pequeno)   [KG/Retalho]                              98.86%
Cebola (local)   [KG/Retalho]                              98.83%
Pimento (verde)   [KG/Retalho]                             98.83%
Feijão (vermelho)   [KG/Retalho]                           98.54%
Mandioca (seca)   [KG/Retalho] 

In [171]:
food_price = food_price[food_price['PUTP'] == 'Milho (branco)   [KG/Retalho]']

In [172]:
food_price.head()

,provincia,categoria,produto,unidade,tipo_preco,ano,mes,preco,PUTP
1372,Cabo_Delgado,cereais e tubérculos,Milho (branco),KG,Retalho,2000,2,1.14,Milho (branco) [KG/Retalho]
1373,Cabo_Delgado,cereais e tubérculos,Milho (branco),KG,Retalho,2000,3,1.04,Milho (branco) [KG/Retalho]
1374,Cabo_Delgado,cereais e tubérculos,Milho (branco),KG,Retalho,2000,4,1.14,Milho (branco) [KG/Retalho]
1375,Cabo_Delgado,cereais e tubérculos,Milho (branco),KG,Retalho,2000,5,1.09,Milho (branco) [KG/Retalho]
1376,Cabo_Delgado,cereais e tubérculos,Milho (branco),KG,Retalho,2000,6,1.04,Milho (branco) [KG/Retalho]


In [173]:
food_price.drop(['categoria', 'produto', 'unidade', 'tipo_preco', 'PUTP'], axis=1, inplace=True)

In [174]:
# Renomear colunas para português
food_price = food_price.rename(columns={
    'preco': 'preco_milho'
})

# Resetar index
food_price = food_price.reset_index(drop=True)

In [175]:
food_price.head()

,provincia,ano,mes,preco_milho
0,Cabo_Delgado,2000,2,1.14
1,Cabo_Delgado,2000,3,1.04
2,Cabo_Delgado,2000,4,1.14
3,Cabo_Delgado,2000,5,1.09
4,Cabo_Delgado,2000,6,1.04


In [176]:
food_price.shape

(2605, 4)